# 06 Cyclic-permutation re-evaluation, item-level split

Each item with K options is presented K times, with the options rotated so that the gold answer
occupies every position once (two rotations for two-option items, five for five-option items). The
base model and the five item-level adapters are evaluated; adapters are loaded with unsloth
`FastLanguageModel.from_pretrained`, which reproduces the released predictions (notebook 03), and
their hashes are checked first. One model is loaded at a time.

For each rotation the chosen option is also recorded in original-position terms, so that
consistency across rotations can be measured, together with the letter margin.

The unrotated predictions (shift 0) of seeds 42-46 are not rerun: they are the
`repro_unslothload_seed<N>_<benchmark>.csv` files of notebook 03, produced with the same loading.
The base model is run from shift 0, and its shift-0 predictions are compared with `baseline_*.csv`.

Inferences per adapter (shifts 1..K-1): 12,567; with the base model (all shifts) about 80,000 in
total, roughly 10 hours on an L4. Each model x benchmark file is checkpointed every 500 inferences.

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, hashlib, platform, string
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
transformers.logging.set_verbosity_error()   # hide max_new_tokens/max_length warnings (no effect on results)
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
REPRO_DIR   = f'{DRIVE_DIR}/repro_check'
OUT_DIR     = f'{DRIVE_DIR}/cyclic_itemlevel'
os.makedirs(OUT_DIR, exist_ok=True)

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LETTERS = list(string.ascii_uppercase)

# adapter hashes; stop if a different adapter is found
EXPECTED_HASH = {42: '1de40b6b5c101d83', 43: '8f3570a519304ca5', 44: '96dbca35b3aa2b68',
                 45: '763ef681101c2872', 46: 'e1fad1151d5af492'}
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]
ADAPTERS = {s: f'{RESULTS_DIR}/adapter_seed{s}' for s in SEEDS}
for s, p in ADAPTERS.items():
    h = sha16(f'{p}/adapter_model.safetensors')
    assert h == EXPECTED_HASH[s], f'seed{s} adapter hash mismatch: {h}'
print('five adapter hashes confirmed')

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_user_prompt(story, question, options):
    lines = [f'{l}. {t}' for l, t in options]
    return f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines)

def parse_options(user_prompt):
    m = re.search(r'\[Candidate Answers\]\n(.+)', user_prompt, re.S)
    block = m.group(1) if m else user_prompt
    return [(l, t.strip()) for l, t in re.findall(r'^([A-E])\.\s*(.+?)$', block, re.M)]

def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        items = _json.load(f)
    for it in items:
        it['options'] = parse_options(it['user_prompt'])
        n = len(it['options'])
        assert [l for l, _ in it['options']] == LETTERS[:n] and it['answer'] in LETTERS[:n], it['item_id']
        # the rebuilt unrotated prompt must equal the original prompt character for character
        assert build_user_prompt(it['story'], it['question'], it['options']) == it['user_prompt'], it['item_id']
        it['n_options'] = n
    return items

MANIFESTS = {bm: load_manifest(bm) for bm in BENCHMARKS}
for bm in BENCHMARKS:
    print(bm, len(MANIFESTS[bm]), pd.Series([it['n_options'] for it in MANIFESTS[bm]]).value_counts().sort_index().to_dict())

def cyclic_shift(options, k):
    # moves the option at original position i to position (i+k) mod n; remap: original letter -> new letter
    n = len(options)
    texts = [t for _, t in options]
    shifted = texts[-k % n:] + texts[:-k % n] if k % n else texts
    new_options = [(LETTERS[i], shifted[i]) for i in range(n)]
    remap = {LETTERS[i]: LETTERS[(i + k) % n] for i in range(n)}
    return new_options, remap

# self-check: for every k the gold option text must sit at the new gold letter
for it in MANIFESTS['HiToM'][:20] + MANIFESTS['OpenToM'][:20] + MANIFESTS['ToMBench'][:50]:
    gold_text = dict(it['options'])[it['answer']]
    for k in range(it['n_options']):
        opts, remap = cyclic_shift(it['options'], k)
        assert dict(opts)[remap[it['answer']]] == gold_text
print('cyclic_shift self-check passed')

@torch.no_grad()
def infer_one(model, tokenizer, user_prompt, letter_ids, lid):
    inputs = tokenizer(to_text(tokenizer, user_prompt), return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id,
                         output_scores=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
    gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
    margin = None
    for step, tok in enumerate(gen_ids.tolist()):
        if tok in letter_ids and step < len(out.scores):
            sc = out.scores[step][0].float().cpu()[lid]
            v, _ = torch.sort(sc, descending=True)
            margin = float(v[0] - v[1])
            break
    return extract_letter(gen), gen, margin

def letter_token_ids(tokenizer):
    ids = {}
    for L in 'ABCDE':
        t = tokenizer.encode(L, add_special_tokens=False)
        assert len(t) == 1, f'{L} is not a single token: {t}'
        ids[t[0]] = L
    return ids, torch.tensor(list(ids.keys()))

def out_csv(model_tag, bm):
    return f'{OUT_DIR}/cyclic_{model_tag}_{bm}.csv'

def run_cyclic(model_tag, model, tokenizer, k_start):
    # k_start=1: shift 0 is taken from existing results / k_start=0: all shifts from 0
    FastLanguageModel.for_inference(model)
    letter_ids, lid = letter_token_ids(tokenizer)
    from tqdm.auto import tqdm
    for bm in BENCHMARKS:
        path = out_csv(model_tag, bm)
        rows = pd.read_csv(path).to_dict('records') if os.path.exists(path) else []
        done = {(r['item_id'], int(r['shift_k'])) for r in rows}
        todo = [(it, k) for it in MANIFESTS[bm] for k in range(k_start, it['n_options']) if (it['item_id'], k) not in done]
        if not todo:
            print(f'[skip] {model_tag} {bm}: done ({len(rows)} rows)'); continue
        print(f'{model_tag} {bm}: {len(done)} rows done, {len(todo)} to go')
        t0 = time.time()
        for j, (it, k) in enumerate(tqdm(todo, desc=f'{model_tag}-{bm}')):
            opts, remap = cyclic_shift(it['options'], k)
            inv = {v: o for o, v in remap.items()}   # new letter -> original letter
            prompt = build_user_prompt(it['story'], it['question'], opts)
            pred, raw, margin = infer_one(model, tokenizer, prompt, letter_ids, lid)
            new_gold = remap[it['answer']]
            rows.append({'model': model_tag, 'benchmark': bm, 'item_id': it['item_id'], 'task': it.get('task'),
                         'n_options': it['n_options'], 'shift_k': k, 'gold_orig': it['answer'], 'new_gold': new_gold,
                         'pred': pred, 'pred_orig_letter': inv.get(pred), 'correct': int(pred == new_gold),
                         'letter_margin': margin, 'raw': raw})
            if (j + 1) % 500 == 0:
                pd.DataFrame(rows).to_csv(path, index=False)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'[saved] {model_tag} {bm}: {len(rows)} rows ({(time.time()-t0)/60:.1f} min)')

meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR}/session_meta_{time.strftime("%Y%m%d_%H%M")}.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)

---
## [1] Cyclic-permutation inference (base, then seeds 42-46)

In [ ]:
# -- [1] Cyclic-permutation loop --
# base: no adapter, all shifts from 0
if not all(os.path.exists(out_csv('base', bm)) and
           len(pd.read_csv(out_csv('base', bm))) == sum(it['n_options'] for it in MANIFESTS[bm]) for bm in BENCHMARKS):
    base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    run_cyclic('base', base_model, base_tok, k_start=0)
    del base_model, base_tok; torch.cuda.empty_cache()
else:
    print('[skip] base done')

# seeds 42-46: same unsloth loading as the original runs, shifts 1..K-1
for s in SEEDS:
    tag = f'seed{s}'
    need = sum(it['n_options'] - 1 for bm in BENCHMARKS for it in MANIFESTS[bm])
    have = sum(len(pd.read_csv(out_csv(tag, bm))) for bm in BENCHMARKS if os.path.exists(out_csv(tag, bm)))
    if have >= need:
        print(f'[skip] {tag} done'); continue
    model, tok = FastLanguageModel.from_pretrained(model_name=ADAPTERS[s], max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    run_cyclic(tag, model, tok, k_start=1)
    del model, tok; torch.cuda.empty_cache()
print('\n[1] all done')

---
## [2] Summary (no GPU needed)

- `acc_orig`: accuracy in the original order (shift 0)
- `lenient`: accuracy averaged over the K rotations
- `strict`: share of items answered correctly in all K rotations
- `consistent`: share of items for which the same option content is chosen in all K rotations
- the tuned - base change of each measure, five-seed mean ± SD and between-seed 95% t-interval
- the same by option count, by OpenToM subtask, and the chosen-letter distribution over all rotations

`code/analyze_cyclic.py` recomputes these tables from the files released in `results/`.

In [ ]:
# -- [2-1] Collect the data and check shift 0 --
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
pd.set_option('display.width', 220)
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
REPRO_DIR   = f'{DRIVE_DIR}/repro_check'
OUT_DIR     = f'{DRIVE_DIR}/cyclic_itemlevel'
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODELS = ['base'] + [f'seed{s}' for s in SEEDS]

frames = []
for m in MODELS:
    for bm in BENCHMARKS:
        p = f'{OUT_DIR}/cyclic_{m}_{bm}.csv'
        if not os.path.exists(p):
            print(f'[missing] {m} {bm}'); continue
        frames.append(pd.read_csv(p))
cyc = pd.concat(frames, ignore_index=True)

# n_options and task come from the base results (all shifts)
info = cyc[cyc['model'] == 'base'].drop_duplicates(['benchmark', 'item_id'])[['benchmark', 'item_id', 'task', 'n_options', 'gold_orig']]

k0 = []
for s in SEEDS:
    for bm in BENCHMARKS:
        p = f'{REPRO_DIR}/repro_unslothload_seed{s}_{bm}.csv'
        if not os.path.exists(p): print(f'[missing] shift 0 seed{s} {bm}'); continue
        d = pd.read_csv(p)
        d = d.merge(info[info['benchmark'] == bm], on='item_id', how='left')
        assert d['n_options'].notna().all(), f'seed{s} {bm}: item_id not in the base results'
        k0.append(pd.DataFrame({'model': f'seed{s}', 'benchmark': bm, 'item_id': d['item_id'], 'task': d['task'],
                                'n_options': d['n_options'].astype(int), 'shift_k': 0, 'gold_orig': d['answer'],
                                'new_gold': d['answer'], 'pred': d['pred'], 'pred_orig_letter': d['pred'],
                                'correct': d['correct'], 'letter_margin': d['letter_margin'], 'raw': d['raw']}))
allr = pd.concat([cyc] + k0, ignore_index=True)

# completeness: exactly one row for each shift 0..K-1 per model and item
cnt = allr.groupby(['model', 'benchmark', 'item_id']).agg(n=('shift_k', 'nunique'), K=('n_options', 'first'))
bad = cnt[cnt['n'] != cnt['K']]
print('items with missing rotations:', len(bad))
if len(bad): print(bad.groupby(['model', 'benchmark']).size())

# shift-0 check: base must equal the released baseline, seed 42 the released CSVs
for tag, ref in [('base', 'baseline_{bm}.csv'), ('seed42', 'finetuned_seed42_{bm}.csv')]:
    for bm in BENCHMARKS:
        a = allr[(allr['model'] == tag) & (allr['benchmark'] == bm) & (allr['shift_k'] == 0)]
        r = pd.read_csv(f'{RESULTS_DIR}/{ref.format(bm=bm)}')
        mg = a.merge(r[['item_id', 'pred']], on='item_id', suffixes=('', '_ref'))
        print(f'shift 0 {tag} {bm}: mismatches with released {int((mg["pred"] != mg["pred_ref"]).sum())}/{len(mg)}')
allr.to_csv(f'{OUT_DIR}/cyclic_all_long.csv', index=False)

In [ ]:
# -- [2-2] Measures by model and benchmark, and change from base --
def item_metrics(g):
    return pd.Series({'acc_orig': g.loc[g['shift_k'] == 0, 'correct'].mean(),
                      'lenient': g['correct'].mean(),
                      'strict': float(g['correct'].min() == 1),
                      'consistent': float(g['pred_orig_letter'].nunique() == 1)})

per_item = allr.groupby(['model', 'benchmark', 'item_id', 'n_options']).apply(item_metrics).reset_index()
per_item.to_csv(f'{OUT_DIR}/cyclic_per_item.csv', index=False)

def summarize(df, keys):
    s = df.groupby(keys)[['acc_orig', 'lenient', 'strict', 'consistent']].mean() * 100
    s['n_items'] = df.groupby(keys).size()
    return s

summ = summarize(per_item, ['model', 'benchmark'])
print('[model x benchmark] accuracy and consistency (%)')
print(summ.round(2).unstack('model').swaplevel(axis=1).sort_index(axis=1).to_string())

base = summ.xs('base', level='model')
delta = []
for s in SEEDS:
    d = summ.xs(f'seed{s}', level='model')[['acc_orig', 'lenient', 'strict', 'consistent']] - base[['acc_orig', 'lenient', 'strict', 'consistent']]
    d['seed'] = s; delta.append(d.reset_index())
delta = pd.concat(delta)
delta.to_csv(f'{OUT_DIR}/cyclic_delta_by_seed.csv', index=False)

agg = delta.groupby('benchmark')[['acc_orig', 'lenient', 'strict', 'consistent']].agg(['mean', 'std', 'count'])
# a direction is stated only when the between-seed 95% interval excludes 0
from scipy.stats import t as _t
for m in ['acc_orig', 'lenient', 'strict', 'consistent']:
    n = agg[(m, 'count')]; half = _t.ppf(0.975, n - 1) * agg[(m, 'std')] / np.sqrt(n)
    agg[(m, 'ci_lo')] = agg[(m, 'mean')] - half; agg[(m, 'ci_hi')] = agg[(m, 'mean')] + half
    agg[(m, 'excl0')] = (agg[(m, 'ci_lo')] > 0) | (agg[(m, 'ci_hi')] < 0)
agg = agg.sort_index(axis=1)
print('\n[tuned - base change, five-seed mean ± SD, 95% interval, pp]')
print(agg.round(2).to_string())
agg.to_csv(f'{OUT_DIR}/cyclic_delta_5seed.csv')

In [ ]:
# -- [2-3] By option count (within-option-count) --
summ_k = summarize(per_item, ['model', 'benchmark', 'n_options'])
bk = summ_k.xs('base', level='model')
dk = []
for s in SEEDS:
    d = summ_k.xs(f'seed{s}', level='model')[['acc_orig', 'lenient', 'strict']] - bk[['acc_orig', 'lenient', 'strict']]
    d['seed'] = s; dk.append(d.reset_index())
dk = pd.concat(dk)
out = dk.groupby(['benchmark', 'n_options'])[['acc_orig', 'lenient', 'strict']].agg(['mean', 'std']).round(2)
out['n_items'] = bk['n_items']
print('[tuned - base change by option count, five-seed mean ± SD, pp]')
print(out.to_string())
out.to_csv(f'{OUT_DIR}/cyclic_delta_by_noptions.csv')

# OpenToM by subtask (the two-option location-cg items were not shuffled originally, so Yes was always A)
po = per_item[per_item['benchmark'] == 'OpenToM'].merge(
    allr[allr['benchmark'] == 'OpenToM'].drop_duplicates(['item_id'])[['item_id', 'task']], on='item_id')
print('\n[OpenToM accuracy by subtask (%)]')
print((po.groupby(['task', 'model'])[['acc_orig', 'lenient', 'strict']].mean() * 100).round(2).unstack('model').to_string())

In [ ]:
# -- [2-4] Distribution of chosen letters over all rotations (letter prior) --
# over a full set of rotations every gold letter occurs equally often, so any deviation of the chosen-letter distribution from uniform is a pure position preference
rows = []
for (m, bm, K), g in allr.groupby(['model', 'benchmark', 'n_options']):
    dist = g['pred'].value_counts(normalize=True) * 100
    row = {'model': m, 'benchmark': bm, 'n_options': K, 'n': len(g),
           'max_minus_min_pp': dist.reindex(list('ABCDE')[:K], fill_value=0).max() - dist.reindex(list('ABCDE')[:K], fill_value=0).min()}
    for L in 'ABCDE'[:K]:
        row[L] = dist.get(L, 0.0)
    rows.append(row)
lp = pd.DataFrame(rows)
lp.to_csv(f'{OUT_DIR}/cyclic_letter_prior.csv', index=False)
print('[chosen-letter distribution (%), all rotations]  uniform = 100/K')
print(lp.round(1).to_string(index=False))